In [1]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np

In [2]:
# Clone or update our BMEN 600 group's GitHub repository.
repo_path = Path("/content/BMEN-600-Group-19")

if repo_path.exists():
    !git -C /content/BMEN-600-Group-19 pull origin main
else:
    !git clone https://github.com/chelseataylor1/BMEN-600-Group-19.git /content/BMEN-600-Group-19

Cloning into '/content/BMEN-600-Group-19'...
remote: Enumerating objects: 256, done.
remote: Counting objects: 100% (256/256), done.
remote: Compressing objects: 100% (238/238), done.
remote: Total 256 (delta 86), reused 5 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (256/256), 133.33 MiB | 19.92 MiB/s, done.
Resolving deltas: 100% (86/86), done.


In [3]:
# Locate the output dataset from Notebook 2.
zip_path = repo_path / "data/processed/OutputFromNotebook2_EHG_Dataset.zip"

# Extract the dataset.
with zipfile.ZipFile(zip_path, "r") as zip_file:
    zip_file.extractall("/content")

# Define the locations of the extracted files.
dataset_path = Path("/content/BMEN600_Matched_Trimmed_EHG_Dataset")
recordings_path = dataset_path / "recordings"
metadata_path = dataset_path / "matched_pregnancies.csv"

print("Dataset extracted successfully!")
print("Recordings:", recordings_path)
print("Metadata:", metadata_path)

Dataset extracted successfully!
Recordings: /content/BMEN600_Matched_Trimmed_EHG_Dataset/recordings
Metadata: /content/BMEN600_Matched_Trimmed_EHG_Dataset/matched_pregnancies.csv


In [4]:
# Load the Matched Pregnancy Metadata.
df_matched = pd.read_csv(metadata_path)

# Confirm the metadata loaded successfully.
print("Matched pregnancies loaded:", len(df_matched))

# Preview the metadata.
display(df_matched.head())

Matched pregnancies loaded: 35


,pregnancy_id,age,early_id,early_week,weight_early,later_id,later_week,weight_later,height,placenta,gestation,newborn_weight,outcome,weeks_between
0,17,22,620,23.9,73,1529,31.9,81,168,end,41.1,3930,induced,8.0
1,18,38,656,22.4,77,1385,31.1,82,167,end,41.3,3710,induced,8.7
2,19,26,665,23.1,83,664,30.4,84,165,front,37.1,2860,induced,7.3
3,20,32,757,22.4,68,756,30.6,72,162,end,39.7,3500,induced,8.2
4,21,29,774,25.0,77,775,31.1,83,171,front,41.6,3180,induced,6.1


In [5]:
# Verify the trimmed EHG recordings.

# Find all trimmed EHG recording CSV files.
recording_files = sorted(recordings_path.glob("*.csv"))

# Confirm the number of recordings.
print("Trimmed EHG recordings loaded:", len(recording_files))

# Preview the contents of one recording.
if recording_files:
    recording = pd.read_csv(recording_files[0])

    print("Recording:", recording_files[0].stem)
    print("Available variables:", recording.columns.tolist())
    print("Number of samples:", len(recording))

    display(recording.head())
else:
    print("No CSV recordings found. Check the dataset path.")

Trimmed EHG recordings loaded: 70
Recording: icehg1010
Available variables: ['trimmed_time', 'trimmed_S1', 'trimmed_S2', 'trimmed_S3', 'trimmed_S4']
Number of samples: 29199


,trimmed_time,trimmed_S1,trimmed_S2,trimmed_S3,trimmed_S4
0,2.500833,0.007172,0.021744,-0.000076,-0.014649
1,2.501667,0.007172,0.021973,-0.001678,-0.016480
2,2.502500,0.006027,0.020600,-0.002518,-0.017090
3,2.503333,0.001221,0.015412,0.000229,-0.013962
4,2.504167,-0.004349,0.008927,0.004730,-0.008545


In [6]:
"""
Median frequency (MF) of EHG channels S1-S4 for every matched pregnancy
with outcome = "cesarean" (excluding early_id 1393).

For each pregnancy the early (~23 wk) and later (~31 wk) recordings are
loaded, and MF is computed for S1, S2, S3 and S4. Output: one table with a
row per pregnancy (early MF, later MF and the change later - early).

Method: each trimmed, filtered recording is converted to a power spectrum
(whole recording, no windows), and the median frequency is the frequency that
splits the power between 0.3 and 3 Hz into two equal halves. The signals are
used as provided (no extra filtering here).

Change columns:
  *_change          = later MF - early MF (Hz)
  *_change_per_week = *_change / weeks_between (Hz per week), where
                      weeks_between = later_week - early_week for that pair

Packages: numpy, pandas, scipy
"""

import os
import numpy as np
import pandas as pd
from scipy.signal import periodogram

# ----------------------------------------------------------------------------
# SETTINGS - change DATA_DIR to where BMEN600_Matched_Trimmed_EHG_Dataset is
# (e.g. a Google Drive path in Colab)
# ----------------------------------------------------------------------------
DATA_DIR = "/content/BMEN600_Matched_Trimmed_EHG_Dataset"
OUTCOME = "cesarean"            # induced | cesarean | induced-cesarean
CHANNELS = ["S1", "S2", "S3", "S4"]

BAND = (0.3, 3.0)      # Hz, band in which the median frequency is calculated

OUT_CSV = f"median_frequency_{OUTCOME}.csv"


def power_spectrum(x, fs):
    """Power spectrum of the whole trimmed, filtered recording (one FFT-based
    periodogram, no windowing into segments; Hann taper, mean removed)."""
    x = np.asarray(x, dtype=float)
    return periodogram(x, fs=fs, window="hann", detrend="constant",
                       scaling="density")


def median_frequency(x, fs, band=BAND):
    """Frequency (Hz) that splits the power spectrum within `band` into two
    halves of equal power (linear interpolation of the cumulative power)."""
    freqs, psd = power_spectrum(x, fs)

    keep = (freqs >= band[0]) & (freqs <= band[1])
    freqs, psd = freqs[keep], psd[keep]

    cum = np.cumsum(psd)
    cum = cum / cum[-1]
    return float(np.interp(0.5, cum, freqs))


def load_recording(rec_id):
    # Locate the CSV recording.
    path = os.path.join(
        DATA_DIR, "recordings", f"icehg{int(rec_id)}.csv"
    )

    # Load the processed EHG recording.
    data = pd.read_csv(path)

    # Retrieve the sampling frequency.
    if "fs" in data.columns:
        fs = float(data["fs"].iloc[0])
    else:
        # Calculate sampling frequency from time in minutes.
        fs = 1 / (np.median(np.diff(data["trimmed_time"])) * 60)

    # Extract the four EHG channels as NumPy arrays.
    channels = {
        "S1": data["trimmed_S1"].to_numpy(),
        "S2": data["trimmed_S2"].to_numpy(),
        "S3": data["trimmed_S3"].to_numpy(),
        "S4": data["trimmed_S4"].to_numpy()
    }

    return channels, fs


def mf_for_recording(rec_id):
    channels, fs = load_recording(rec_id)
    return {ch: median_frequency(sig, fs) for ch, sig in channels.items()}

# Calculating the Change in Median Frequency per Weekly Recordings: 23 & 31 Weeks (Jerold)

**Change in Median Frequency between early (23 weeks) & later (31 weeks) EHG recordings across ALL pregnancies**

In [7]:
# Ensure df_cesarean_mf exists by warning the user if it is missing
try:
    table = df_cesarean_mf

    # Extract value columns for calculation
    mf_cols = ['S1_early_MF', 'S1_later_MF', 'S1_change', 'S1_change_per_week',
               'S2_early_MF', 'S2_later_MF', 'S2_change', 'S2_change_per_week',
               'S3_early_MF', 'S3_later_MF', 'S3_change', 'S3_change_per_week',
               'S4_early_MF', 'S4_later_MF', 'S4_change', 'S4_change_per_week']

    # Generate summary table for mean, median, and std
    summary_stats = table[mf_cols].agg(['mean', 'median', 'std']).round(4)
    print(summary_stats.T)
except NameError:
    print("Error: 'df_cesarean_mf' is not defined. Please run cell P5pNK4JKWb99 first to calculate individual median frequencies!")

Error: 'df_cesarean_mf' is not defined. Please run cell P5pNK4JKWb99 first to calculate individual median frequencies!


**Change in Median Frequency between early (23 weeks) & later (31 weeks) EHG recordings across individual pregnancies**

In [8]:
# 1. Filter for the target outcome group (cesarean) and exclude early_id 1393
df_group = df_matched[
    (df_matched["outcome"] == "cesarean") & (df_matched["early_id"] != 1393)
].copy()

individual_results = []

for _, row in df_group.iterrows():
    preg_id = int(row["pregnancy_id"])
    weeks_between = float(row["weeks_between"])

    # Load and calculate early (~23 wk) and later (~31 wk) MF for channels S1-S4
    early_mf = mf_for_recording(row["early_id"])
    later_mf = mf_for_recording(row["later_id"])

    preg_data = {
        "pregnancy_id": preg_id,
        "early_week": row["early_week"],
        "later_week": row["later_week"],
        "weeks_between": weeks_between,
    }

    # Compute absolute change and weekly rate per channel
    for ch in CHANNELS:
        change = later_mf[ch] - early_mf[ch]
        preg_data[f"{ch}_early_MF"] = round(early_mf[ch], 4)
        preg_data[f"{ch}_later_MF"] = round(later_mf[ch], 4)
        preg_data[f"{ch}_change"] = round(change, 4)
        preg_data[f"{ch}_change_per_week"] = round(
            change / weeks_between if weeks_between > 0 else 0.0, 4
        )

    individual_results.append(preg_data)

# 2. Convert to DataFrame and display individual results
df_cesarean_mf = pd.DataFrame(individual_results)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
display(df_cesarean_mf)

,pregnancy_id,early_week,later_week,weeks_between,S1_early_MF,S1_later_MF,S1_change,S1_change_per_week,S2_early_MF,S2_later_MF,S2_change,S2_change_per_week,S3_early_MF,S3_later_MF,S3_change,S3_change_per_week,S4_early_MF,S4_later_MF,S4_change,S4_change_per_week
0,65,23.0,30.3,7.3,0.3269,0.3591,0.0322,0.0044,0.3356,0.3527,0.0171,0.0023,0.3713,0.3698,-0.0014,-0.0002,0.3406,0.4395,0.0989,0.0135
1,66,24.9,30.6,5.7,1.0183,0.7190,-0.2994,-0.0525,0.4330,0.5070,0.0741,0.0130,1.0352,0.8679,-0.1673,-0.0293,0.5647,0.4138,-0.1509,-0.0265
2,67,23.1,30.9,7.8,0.4469,0.4497,0.0028,0.0004,0.4602,0.3953,-0.0649,-0.0083,0.4383,0.3652,-0.0732,-0.0094,0.4418,0.4467,0.0049,0.0006
3,68,23.3,30.7,7.4,0.4268,0.4278,0.0010,0.0001,0.4000,0.4000,0.0000,0.0000,0.4107,0.4050,-0.0056,-0.0008,0.4072,0.4927,0.0855,0.0116
4,69,22.1,31.0,8.9,0.4123,0.4507,0.0384,0.0043,0.3817,0.4026,0.0209,0.0024,1.2519,0.5140,-0.7379,-0.0829,1.0457,0.5014,-0.5443,-0.0612


**Based on the analysis from your Cesarean_medfreq.ipynb - Colab notebook, yes, there is a consistent downward trend in Median Frequency (MF) across late gestation for cesarean pregnancies, though the degree of shift varies across electrode locations.Across the 5 matched cesarean pregnancies ($N=5$) tracked from early (23 weeks) to later (31 weeks) gestation, the group summary statistics show the following key patterns:**

1. Overrall Decreasing Trend across S1 to S4 from week 23 to 31
2. Channels S3 & S4 showing largest overall drop in median frequency: S3 (-0.039 Hz) & S4 (-0.073 Hz)
3. Reduction in Frequency Variance: a High baseline in early pregnancy, narrowed by 31 weeks, indicating uterine burst spectra consolidation into tighter, low-frequency bands as gestational progression towards full-term.
- electrical activity in myometrium evolves from disorganized, broad-frequency baseline firing into synchronized, regular, and lower-frequency electrical bursts as pregnancy advances.

Now we need to save this dataset and add it onto github

In [9]:
import shutil
from pathlib import Path
from google.colab import files

# 1. Copy the original dataset into a new folder
output_folder = Path("/content/BMEN600_MF_Dataset")
shutil.copytree(dataset_path, output_folder, dirs_exist_ok=True)

# 2. Add the new median frequency variables to the original metadata
df_updated = df_matched.merge(
    df_cesarean_mf,
    on="pregnancy_id",
    how="left"
)

# 3. Save the updated CSV
df_updated.to_csv(output_folder / "matched_pregnancies.csv", index=False)

# 4. Create the ZIP file
zip_path = shutil.make_archive(
    "/content/BMEN600_MF_Dataset",
    "zip",
    root_dir="/content",
    base_dir="BMEN600_MF_Dataset"
)

# 5. Download the ZIP
print("ZIP created successfully!")
files.download(zip_path)

ZIP created successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>